1. Реалізуйте функцію harmonic_with_noise, яка приймає наступні параметри:
amplitude - амплітуда гармоніки.
frequency - частота гармоніки.
phase – фазовий зсув гармоніки
noise_mean - середнє значення шуму.
noise_covariance – дисперсія шуму
show_noise - флаг, який вказує, чи слід показувати шум на графіку.

2. У програмі має бути створено головне вікно з такими елементами інтерфейсу:
Поле для графіка функції (plot) (або декілька полів)
Слайдери (sliders), які відповідають за амплітуду, частоту гармоніки, а також слайдери для параметрів шуму
Чекбокс для перемикання відображення шуму на гармоніці
Кнопка «Reset», яка відновлює початкові параметри

3. Програма повинна мати початкові значення кожного параметру, а також передавати  параметри для відображення оновленого графіка.

4. Через чекбокс користувач може вмикати або вимикати відображення шуму на графіку. Якщо прапорець прибрано – відображати «чисту гармоніку», якщо ні – зашумлену.

   
6. Після оновлення параметрів програма повинна одразу оновлювати графік функції гармоніки з накладеним шумом згідно з виставленими параметрами.

7. Після натискання кнопки «Reset», мають відновитись початкові параметри
Отриману гармоніку з накладеним на неї шумом відфільтруйте за допомогою фільтру на ваш вибір. Відфільтрована гармоніка має бути максимально близька до «чистої»

8. Відобразіть відфільтровану «чисту» гармоніку поряд з початковою

9. Додайте відповідні інтерактивні елементи (чекбокс показу, параметри фільтра тощо) та оновіть наявні: відфільтрована гармоніка має оновлюватись разом з початковою.

10. Залиште інструкції для користувача, які пояснюють, як користуватися програмою.


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, CheckButtons
from scipy.signal import butter, filtfilt

class HarmonicVisualizer:
    def __init__(self):

        #Початкові параметри
        self.init_amp = 1.0
        self.init_freq = 1.0
        self.init_phase = 0.0
        self.init_noise_mean = 0.0
        self.init_noise_cov = 0.5
        self.init_cutoff = 2.0  
        
        #Часові налаштування 
        self.fs = 100.0
        self.t = np.linspace(0, 10, int(10 * self.fs))
        
        #Збереження шуму
        self.current_noise_params = (self.init_noise_mean, self.init_noise_cov)
        self.noise = np.random.normal(self.init_noise_mean, np.sqrt(self.init_noise_cov), len(self.t))
        
        self.setup_gui()

    def harmonic_with_noise(self, amplitude, frequency, phase, noise_mean, noise_covariance):
        #Чиста гармоніка
        y_clean = amplitude * np.sin(2 * np.pi * frequency * self.t + phase)
        
        #Встановлкння нового шуму при зміні параметрів
        if (noise_mean, noise_covariance) != self.current_noise_params:

            self.noise = np.random.normal(noise_mean, np.sqrt(noise_covariance), len(self.t))
            self.current_noise_params = (noise_mean, noise_covariance)
            
        y_noisy = y_clean + self.noise
        return y_clean, y_noisy

    def apply_filter(self, data, cutoff_freq):
        #Фільтр низьких частот Баттерворта
        nyq = 0.5 * self.fs
        normal_cutoff = cutoff_freq / nyq
        
        #Для корекції значень зрізу
        if normal_cutoff >= 1.0: normal_cutoff = 0.99
        if normal_cutoff <= 0: normal_cutoff = 0.01
            
        b, a = butter(4, normal_cutoff, btype='low', analog=False)
        y_filtered = filtfilt(b, a, data)
        return y_filtered

    def setup_gui(self):
        #Налаштування слайдерів
        self.fig, self.ax = plt.subplots(figsize=(10, 7))
        plt.subplots_adjust(left=0.1, bottom=0.5)
        
        self.ax.set_title('Гармоніка з шумом та фільтрацією')
        self.ax.set_xlabel('Час (t)')
        self.ax.set_ylabel('Амплітуда (y)')
        self.ax.grid(True)

        #Отримання початкових даних
        y_clean, y_noisy = self.harmonic_with_noise(
            self.init_amp, self.init_freq, self.init_phase, self.init_noise_mean, self.init_noise_cov
        )
        y_filtered = self.apply_filter(y_noisy, self.init_cutoff)

        #Відображення ліній
        self.line_clean, = self.ax.plot(self.t, y_clean, label='Чиста гармоніка', color='green', lw=2)
        self.line_noisy, = self.ax.plot(self.t, y_noisy, label='Зашумлена', color='red', alpha=0.5, visible=False)
        self.line_filtered, = self.ax.plot(self.t, y_filtered, label='Відфільтрована', color='blue', lw=2, visible=False)
        self.ax.legend(loc='upper right')

        #Створення слайдерів
        ax_amp = plt.axes([0.15, 0.40, 0.65, 0.03])
        ax_freq = plt.axes([0.15, 0.35, 0.65, 0.03])
        ax_phase = plt.axes([0.15, 0.30, 0.65, 0.03])
        ax_n_mean = plt.axes([0.15, 0.25, 0.65, 0.03])
        ax_n_cov = plt.axes([0.15, 0.20, 0.65, 0.03])
        ax_cutoff = plt.axes([0.15, 0.15, 0.65, 0.03])

        self.s_amp = Slider(ax_amp, 'Амплітуда', 0.1, 5.0, valinit=self.init_amp)
        self.s_freq = Slider(ax_freq, 'Частота', 0.1, 10.0, valinit=self.init_freq)
        self.s_phase = Slider(ax_phase, 'Фаза', 0.0, 2*np.pi, valinit=self.init_phase)
        self.s_n_mean = Slider(ax_n_mean, 'Шум (Середнє)', -2.0, 2.0, valinit=self.init_noise_mean)
        self.s_n_cov = Slider(ax_n_cov, 'Шум (Дисперсія)', 0.0, 5.0, valinit=self.init_noise_cov)
        self.s_cutoff = Slider(ax_cutoff, 'Зріз фільтра', 0.1, 20.0, valinit=self.init_cutoff)

        #Чекбокси
        ax_check = plt.axes([0.85, 0.25, 0.13, 0.15])
        self.check = CheckButtons(ax_check, ['Показати шум', 'Показати фільтр'], [False, False])

        #Ресетка
        ax_reset = plt.axes([0.85, 0.15, 0.1, 0.04])
        self.btn_reset = Button(ax_reset, 'Reset', hovercolor='0.975')

        #Зв'язування подій
        self.s_amp.on_changed(self.update)
        self.s_freq.on_changed(self.update)
        self.s_phase.on_changed(self.update)
        self.s_n_mean.on_changed(self.update)
        self.s_n_cov.on_changed(self.update)
        self.s_cutoff.on_changed(self.update)
        self.check.on_clicked(self.toggle_visibility)
        self.btn_reset.on_clicked(self.reset)

        plt.show()

    def update(self, val):
        #Отримуємо нові значення
        amp = self.s_amp.val
        freq = self.s_freq.val
        phase = self.s_phase.val
        n_mean = self.s_n_mean.val
        n_cov = self.s_n_cov.val
        cutoff = self.s_cutoff.val
        #Оновлюємо данні
        y_clean, y_noisy = self.harmonic_with_noise(amp, freq, phase, n_mean, n_cov)
        y_filtered = self.apply_filter(y_noisy, cutoff)
        #Оноалюємо графіки
        self.line_clean.set_ydata(y_clean)
        self.line_noisy.set_ydata(y_noisy)
        self.line_filtered.set_ydata(y_filtered)

        #Зміна осі У
        self.ax.relim()
        self.ax.autoscale_view()
        self.fig.canvas.draw_idle()

    def toggle_visibility(self, label):
        if label == 'Показати шум':
            self.line_noisy.set_visible(not self.line_noisy.get_visible())
        elif label == 'Показати фільтр':
            self.line_filtered.set_visible(not self.line_filtered.get_visible())
        self.fig.canvas.draw_idle()

    def reset(self, event):
        self.s_amp.reset()
        self.s_freq.reset()
        self.s_phase.reset()
        self.s_n_mean.reset()
        self.s_n_cov.reset()
        self.s_cutoff.reset()

if __name__ == '__main__':
    app = HarmonicVisualizer()